In [50]:
import os
import time
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.ensemble import IsolationForest
from IPython.display import display

In [51]:
possible_paths = [
    "features.parquet",
    "data/features.parquet"
]

PARQUET_PATH = next(
    (path for path in possible_paths if os.path.exists(path)),
    None
)
df = pd.read_parquet(PARQUET_PATH)

print("Loaded:", PARQUET_PATH)
print("Shape:", df.shape)

display(df.head())

Loaded: data/features.parquet
Shape: (9781161, 9)


,row_id,Rndrng_NPI,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,log_pay,log_benes,srvcs_per_bene,markup
0,0,1003000126,Internal Medicine,99221,F,4.133187,3.583519,0.000000,1.280051
1,1,1003000126,Internal Medicine,99222,F,4.564422,5.010635,0.000000,1.025149
2,2,1003000126,Internal Medicine,99223,F,4.882797,4.143135,0.000000,1.526492
3,3,1003000126,Internal Medicine,99231,F,3.517683,2.397895,0.374693,0.937340
4,4,1003000126,Internal Medicine,99232,F,4.068802,3.610918,0.637577,1.969501


In [52]:
model_features = [
    "log_pay",
    "log_benes",
    "srvcs_per_bene",
    "markup"
]

peer_levels = {
    1: ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],
    2: ["Rndrng_Prvdr_Type", "HCPCS_Cd"],
    3: ["Rndrng_Prvdr_Type"]
}

print("Model features:")
print(model_features)

print("\nPeer hierarchy:")
for level, cols in peer_levels.items():
    print(f"Level {level}: {cols}")

Model features:
['log_pay', 'log_benes', 'srvcs_per_bene', 'markup']

Peer hierarchy:
Level 1: ['Rndrng_Prvdr_Type', 'HCPCS_Cd', 'Place_Of_Srvc']
Level 2: ['Rndrng_Prvdr_Type', 'HCPCS_Cd']
Level 3: ['Rndrng_Prvdr_Type']


In [53]:
required_cols = ["row_id", "Rndrng_NPI"]

required_cols += list(
    set(
        col
        for cols in peer_levels.values()
        for col in cols
    )
)

required_cols += model_features

missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df_model = (
    df
    .dropna(subset=model_features + peer_levels[1])
    .copy()
)

print("Original rows:", len(df))
print("Rows available for modeling:", len(df_model))

print("\nMissing model values:")
print(
    df_model[model_features]
    .isna()
    .sum()
)

Original rows: 9781161
Rows available for modeling: 9781161

Missing model values:
log_pay           0
log_benes         0
srvcs_per_bene    0
markup            0
dtype: int64


In [54]:
MIN_PEER_SIZE = 100

for level, cols in peer_levels.items():

    group = df_model.groupby(
        cols,
        observed=True
    )
    df_model[f"peer_size_L{level}"] = group["row_id"].transform("size")
    df_model[
        f"peer_id_L{level}"
    ] = group.ngroup()

print(
    df_model[
        [
            "peer_size_L1",
            "peer_size_L2",
            "peer_size_L3"
        ]
    ].describe()
)

       peer_size_L1  peer_size_L2  peer_size_L3
count  9.781161e+06  9.781161e+06  9.781161e+06
mean   1.166055e+04  1.355727e+04  4.832894e+05
std    1.761092e+04  1.962358e+04  3.870712e+05
min    1.000000e+00  1.000000e+00  1.000000e+00
25%    1.153000e+03  1.504000e+03  1.272230e+05
50%    5.176000e+03  6.401000e+03  3.822670e+05
75%    1.296200e+04  1.614900e+04  8.054320e+05
max    9.132600e+04  1.064620e+05  1.119939e+06


In [55]:
df_model["peer_level"] = 0
df_model["peer_id"] = -1
df_model["peer_n"] = 0

for level in [1, 2, 3]:

    use_this_level = (
        (df_model["peer_level"] == 0)
        & (df_model[f"peer_size_L{level}"] >= MIN_PEER_SIZE)
    )

    df_model.loc[use_this_level, "peer_level"] = level

    df_model.loc[use_this_level, "peer_id"] = (
        df_model.loc[use_this_level, f"peer_id_L{level}"]
    )

    df_model.loc[use_this_level, "peer_n"] = (
        df_model.loc[use_this_level, f"peer_size_L{level}"]
    )

print(
    df_model["peer_level"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

peer_level
0     0.004642
1    94.561075
2     0.905138
3     4.529145
Name: proportion, dtype: float64


In [56]:
coverage = (
    df_model["peer_level"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

coverage_table = pd.DataFrame({
    "peer_level": [0, 1, 2, 3],
    "meaning": [
        "Unscored",
        "Provider Type + HCPCS + POS",
        "Provider Type + HCPCS",
        "Provider Type"
    ],
    "percent_of_rows": [
        coverage.get(0, 0),
        coverage.get(1, 0),
        coverage.get(2, 0),
        coverage.get(3, 0)
    ]
})

display(coverage_table)

,peer_level,meaning,percent_of_rows
0,0,Unscored,0.004642
1,1,Provider Type + HCPCS + POS,94.561075
2,2,Provider Type + HCPCS,0.905138
3,3,Provider Type,4.529145


In [57]:
MIN_TEST_GROUP_SIZE = 500
TEST_PEERS_PER_LEVEL = 5
PEER_SAMPLE_SEED = 42

test_peer_specs = []

for level, cols in peer_levels.items():
    size_col = f"peer_size_L{level}"

    used_rows = df_model[
        df_model["peer_level"] == level
    ]

    candidates = (
        used_rows[cols + [size_col]]
        .drop_duplicates()
    )

    candidates = candidates[
        candidates[size_col] >= MIN_TEST_GROUP_SIZE
    ]

    n_sample = min(
        TEST_PEERS_PER_LEVEL,
        len(candidates)
    )

    print(
        f"Level {level}: {len(candidates)} eligible groups"
    )

    if n_sample == 0:
        continue

    sampled = candidates.sample(
        n=n_sample,
        random_state=PEER_SAMPLE_SEED + level
    )

    for _, row in sampled.iterrows():
        peer_key = tuple(
            row[col] for col in cols
        )

        test_peer_specs.append({
            "peer_level": level,
            "peer_key": peer_key,
            "peer_n": int(row[size_col])
        })

test_peer_specs = pd.DataFrame(test_peer_specs)

display(test_peer_specs)

Level 1: 2717 eligible groups
Level 2: 948 eligible groups
Level 3: 94 eligible groups


,peer_level,peer_key,peer_n
0,1,"(Nurse Practitioner, G0439, F)",835
1,1,"(Hematology-Oncology, J9271, O)",1314
2,1,"(Nurse Practitioner, 73502, O)",803
3,1,"(Diagnostic Radiology, 36561, F)",1098
4,1,"(Dermatology, 13151, O)",506
5,2,"(Hospitalist, 99497)",1625
6,2,"(Family Practice, 99443)",968
7,2,"(Dermatology, 10060)",1310
8,2,"(Clinical Laboratory, 83036)",1016
9,2,"(Nurse Practitioner, G0442)",2091


In [58]:
peer_data = []

for _, spec in test_peer_specs.iterrows():
    level = int(spec["peer_level"])
    cols = peer_levels[level]
    key = spec["peer_key"]

    mask = pd.Series(True, index=df_model.index)

    for col, value in zip(cols, key):
        mask &= df_model[col] == value

    group = df_model[mask].copy()

    peer_data.append({
        "peer_level": level,
        "peer_key": key,
        "group": group
    })

print("Test peer groups collected:", len(peer_data))

print("\nGroups by level:")
print(
    pd.Series(
        [x["peer_level"] for x in peer_data]
    ).value_counts()
)

Test peer groups collected: 15

Groups by level:
1    5
2    5
3    5
Name: count, dtype: int64


In [59]:
TRAIN_FRAC = 0.70

SPLIT_SEED = 123

def split_peer(group):

    train = group.sample(
        frac=TRAIN_FRAC,
        random_state=SPLIT_SEED
    )
    evaluation = group.drop(
        index=train.index
    )

    return train, evaluation

In [60]:
def top_k_jaccard(scores_a, scores_b, top_pct=0.05):
    n_top = max(1, int(len(scores_a) * top_pct))

    top_a = set(scores_a.nlargest(n_top).index)
    top_b = set(scores_b.nlargest(n_top).index)

    intersection = len(top_a & top_b)
    union = len(top_a | top_b)

    return intersection / union

In [61]:
configs = [
    {
        "config_id": "BASELINE",
        "n_estimators": 100,
        "max_samples": "auto",
        "max_features": 1.0
    },
    {
        "config_id": "TREES_50",
        "n_estimators": 50,
        "max_samples": "auto",
        "max_features": 1.0
    },
    {
        "config_id": "TREES_200",
        "n_estimators": 200,
        "max_samples": "auto",
        "max_features": 1.0
    },
    {
        "config_id": "TREES_400",
        "n_estimators": 400,
        "max_samples": "auto",
        "max_features": 1.0
    },
    {
        "config_id": "SAMPLES_64",
        "n_estimators": 100,
        "max_samples": 64,
        "max_features": 1.0
    },
    {
        "config_id": "SAMPLES_128",
        "n_estimators": 100,
        "max_samples": 128,
        "max_features": 1.0
    },
    {
        "config_id": "FEATURES_50PCT",
        "n_estimators": 100,
        "max_samples": "auto",
        "max_features": 0.50
    },
    {
        "config_id": "FEATURES_75PCT",
        "n_estimators": 100,
        "max_samples": "auto",
        "max_features": 0.75
    }
]

SEEDS = [1, 42, 100]

print("Configurations:", len(configs))
print("Random seeds:", SEEDS)

Configurations: 8
Random seeds: [1, 42, 100]


In [62]:
def evaluate_config_on_peer(group, config):

    train, evaluation = split_peer(group)

    X_train = train[model_features].copy()
    X_eval = evaluation[model_features].copy()

    seed_scores = {}
    fit_times = []

    for seed in SEEDS:

        model = IsolationForest(
            n_estimators=config["n_estimators"],
            max_samples=config["max_samples"],
            max_features=config["max_features"],
            contamination="auto",
            bootstrap=False,
            random_state=seed,
            n_jobs=-1
        )

        start = time.perf_counter()

        model.fit(X_train)

        fit_times.append(
            time.perf_counter() - start
        )

        scores = -model.score_samples(X_eval)

        seed_scores[seed] = pd.Series(
            scores,
            index=evaluation["row_id"].to_numpy()
        )

    spearman_values = []
    jaccard_values = []

    for seed_a, seed_b in combinations(SEEDS, 2):

        scores_a = seed_scores[seed_a]
        scores_b = seed_scores[seed_b]

        rho, _ = spearmanr(
            scores_a,
            scores_b
        )

        jac = top_k_jaccard(
            scores_a,
            scores_b,
            top_pct=0.05
        )

        spearman_values.append(rho)
        jaccard_values.append(jac)

    return {
        "seed_spearman": np.mean(spearman_values),
        "top5_jaccard": np.mean(jaccard_values),
        "fit_seconds": np.mean(fit_times),
        "train_rows": len(train),
        "eval_rows": len(evaluation)
    }

In [63]:
results = []

total_tests = len(configs) * len(peer_data)
finished = 0

for config in configs:
    print("\nTesting:", config["config_id"])

    for peer_info in peer_data:
        level = peer_info["peer_level"]
        key = peer_info["peer_key"]
        group = peer_info["group"]

        metrics = evaluate_config_on_peer(group, config)

        results.append({
            "config_id": config["config_id"],
            "peer_level": level,
            "peer_key": str(key),
            "n_estimators": config["n_estimators"],
            "max_samples": str(config["max_samples"]),
            "max_features": config["max_features"],
            **metrics
        })

        finished += 1

        if finished % 10 == 0:
            print(f"Finished {finished}/{total_tests}")

results = pd.DataFrame(results)

print("\nDONE")
print("Result rows:", len(results))

display(results.head())


Testing: BASELINE
Finished 10/120

Testing: TREES_50
Finished 20/120
Finished 30/120

Testing: TREES_200
Finished 40/120

Testing: TREES_400
Finished 50/120
Finished 60/120

Testing: SAMPLES_64
Finished 70/120

Testing: SAMPLES_128
Finished 80/120
Finished 90/120

Testing: FEATURES_50PCT
Finished 100/120

Testing: FEATURES_75PCT
Finished 110/120
Finished 120/120

DONE
Result rows: 120


,config_id,peer_level,peer_key,n_estimators,max_samples,max_features,seed_spearman,top5_jaccard,fit_seconds,train_rows,eval_rows
0,BASELINE,1,"('Nurse Practitioner', 'G0439', 'F')",100,auto,1.0,0.969128,0.720147,0.117034,584,251
1,BASELINE,1,"('Hematology-Oncology', 'J9271', 'O')",100,auto,1.0,0.970934,0.839683,0.123101,920,394
2,BASELINE,1,"('Nurse Practitioner', '73502', 'O')",100,auto,1.0,0.974592,0.720147,0.098594,562,241
3,BASELINE,1,"('Diagnostic Radiology', '36561', 'F')",100,auto,1.0,0.982085,0.746589,0.090578,769,329
4,BASELINE,1,"('Dermatology', '13151', 'O')",100,auto,1.0,0.984660,0.833333,0.091959,354,152


In [64]:
summary = (
    results
    .groupby(
        ["config_id", "n_estimators", "max_samples", "max_features"],
        as_index=False
    )
    .agg(
        mean_spearman=("seed_spearman", "mean"),
        median_spearman=("seed_spearman", "median"),
        std_spearman=("seed_spearman", "std"),
        mean_top5_jaccard=("top5_jaccard", "mean"),
        median_top5_jaccard=("top5_jaccard", "median"),
        std_top5_jaccard=("top5_jaccard", "std"),
        mean_fit_seconds=("fit_seconds", "mean")
    )
)

display(
    summary.sort_values(
        ["mean_spearman", "mean_top5_jaccard"],
        ascending=False
    )
)

,config_id,n_estimators,max_samples,max_features,mean_spearman,median_spearman,std_spearman,mean_top5_jaccard,median_top5_jaccard,std_top5_jaccard,mean_fit_seconds
6,TREES_400,400,auto,1.00,0.990398,0.990784,0.003607,0.886478,0.893167,0.072549,0.327420
5,TREES_200,200,auto,1.00,0.982457,0.984682,0.006785,0.868842,0.868692,0.072740,0.153898
0,BASELINE,100,auto,1.00,0.969067,0.970934,0.011115,0.780744,0.779406,0.054115,0.102355
4,SAMPLES_64,100,64,1.00,0.968147,0.972798,0.012099,0.750714,0.777778,0.122084,0.080385
3,SAMPLES_128,100,128,1.00,0.967348,0.968651,0.014096,0.786816,0.812266,0.078731,0.083695
2,FEATURES_75PCT,100,auto,0.75,0.962311,0.962930,0.012062,0.737683,0.722222,0.060340,0.087086
1,FEATURES_50PCT,100,auto,0.50,0.945806,0.952638,0.022861,0.766626,0.754690,0.071338,0.086069
7,TREES_50,50,auto,1.00,0.939523,0.939563,0.018180,0.720274,0.703704,0.097719,0.054824


In [65]:
level_summary = (
    results
    .groupby(
        ["config_id", "peer_level"],
        as_index=False
    )
    .agg(
        mean_spearman=("seed_spearman", "mean"),
        mean_top5_jaccard=("top5_jaccard", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
        number_of_peers=("peer_key", "count")
    )
)

display(level_summary)

,config_id,peer_level,mean_spearman,mean_top5_jaccard,mean_fit_seconds,number_of_peers
0,BASELINE,1,0.976280,0.771980,0.104253,5
1,BASELINE,2,0.958640,0.782190,0.084377,5
2,BASELINE,3,0.972282,0.788063,0.118436,5
3,FEATURES_50PCT,1,0.952997,0.811268,0.077084,5
4,FEATURES_50PCT,2,0.926233,0.752136,0.076111,5
5,FEATURES_50PCT,3,0.958189,0.736474,0.105012,5
6,FEATURES_75PCT,1,0.967512,0.725023,0.078356,5
7,FEATURES_75PCT,2,0.950457,0.736293,0.078271,5
8,FEATURES_75PCT,3,0.968964,0.751734,0.104631,5
9,SAMPLES_128,1,0.972929,0.815163,0.075690,5


In [66]:
tree_configs = [
    "TREES_50",
    "BASELINE",
    "TREES_200",
    "TREES_400"
]
sample_configs = [
    "SAMPLES_64",
    "SAMPLES_128",
    "BASELINE"
]
feature_configs = [
    "FEATURES_50PCT",
    "FEATURES_75PCT",
    "BASELINE"
]

In [67]:
print("N_ESTIMATORS")

display(
    summary[summary["config_id"].isin(tree_configs)]
)

N_ESTIMATORS


,config_id,n_estimators,max_samples,max_features,mean_spearman,median_spearman,std_spearman,mean_top5_jaccard,median_top5_jaccard,std_top5_jaccard,mean_fit_seconds
0,BASELINE,100,auto,1.0,0.969067,0.970934,0.011115,0.780744,0.779406,0.054115,0.102355
5,TREES_200,200,auto,1.0,0.982457,0.984682,0.006785,0.868842,0.868692,0.072740,0.153898
6,TREES_400,400,auto,1.0,0.990398,0.990784,0.003607,0.886478,0.893167,0.072549,0.327420
7,TREES_50,50,auto,1.0,0.939523,0.939563,0.018180,0.720274,0.703704,0.097719,0.054824


In [68]:
print("MAX_SAMPLES")

display(
    summary[summary["config_id"].isin(sample_configs)]
)

MAX_SAMPLES


,config_id,n_estimators,max_samples,max_features,mean_spearman,median_spearman,std_spearman,mean_top5_jaccard,median_top5_jaccard,std_top5_jaccard,mean_fit_seconds
0,BASELINE,100,auto,1.0,0.969067,0.970934,0.011115,0.780744,0.779406,0.054115,0.102355
3,SAMPLES_128,100,128,1.0,0.967348,0.968651,0.014096,0.786816,0.812266,0.078731,0.083695
4,SAMPLES_64,100,64,1.0,0.968147,0.972798,0.012099,0.750714,0.777778,0.122084,0.080385


In [69]:
print("MAX_FEATURES")

display(
    summary[summary["config_id"].isin(feature_configs)]
)

MAX_FEATURES


,config_id,n_estimators,max_samples,max_features,mean_spearman,median_spearman,std_spearman,mean_top5_jaccard,median_top5_jaccard,std_top5_jaccard,mean_fit_seconds
0,BASELINE,100,auto,1.00,0.969067,0.970934,0.011115,0.780744,0.779406,0.054115,0.102355
1,FEATURES_50PCT,100,auto,0.50,0.945806,0.952638,0.022861,0.766626,0.754690,0.071338,0.086069
2,FEATURES_75PCT,100,auto,0.75,0.962311,0.962930,0.012062,0.737683,0.722222,0.060340,0.087086


In [70]:
print("TREE SETTINGS BY PEER LEVEL")

display(
    level_summary[
        level_summary["config_id"].isin(tree_configs)
    ].sort_values(
        ["peer_level", "config_id"]
    )
)

TREE SETTINGS BY PEER LEVEL


,config_id,peer_level,mean_spearman,mean_top5_jaccard,mean_fit_seconds,number_of_peers
0,BASELINE,1,0.976280,0.771980,0.104253,5
15,TREES_200,1,0.985752,0.889121,0.140038,5
18,TREES_400,1,0.992253,0.922629,0.254903,5
21,TREES_50,1,0.948921,0.711811,0.053923,5
1,BASELINE,2,0.958640,0.782190,0.084377,5
16,TREES_200,2,0.976204,0.842427,0.136613,5
19,TREES_400,2,0.986810,0.863755,0.354621,5
22,TREES_50,2,0.928329,0.773536,0.050922,5
2,BASELINE,3,0.972282,0.788063,0.118436,5
17,TREES_200,3,0.985416,0.874980,0.185044,5
